In [1]:
!pip install keras_facenet
import tensorflow as tf
from keras_facenet import FaceNet
import numpy as np
import pandas as pd

In [2]:
def resize_to_160(img):
    """
    img: Tensor or numpy array of shape (256, 256, 3)
    returns: Tensor (160, 160, 3)
    """
    img = tf.image.resize(img, (160, 160), method='bilinear')
    img = img.numpy()
    return img

In [3]:
def get_embedding(img):
  """
  The embedder.embeddings function expects a list of numpy arrays.
  img is already a numpy array from resize_to_160, so we wrap it in a list.
  """
  img = resize_to_160(img)
  emb = embedder.embeddings([img])[0]
  #emb = np.expand_dims(emb, axis=0)
  return emb

In [4]:
def process_row(row):
    row["embedding"] = resize_to_160(row["image"])
    return row

In [5]:
def get_embeddings(images):
  images = np.array(images['image'])
  embeddings = embedder.embeddings(images)
  #emb = np.expand_dims(emb, axis=0)
  return {"embedding": embeddings}

In [6]:
from datasets import load_dataset

ds = load_dataset("tonyassi/celebrity-1000")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
ds['train'].shape

(18184, 2)

In [8]:
embedder = FaceNet()

In [9]:
ds = ds['train']

In [10]:
ds.shape

(18184, 2)

In [11]:
ds = ds.map(process_row)

In [12]:
ds = ds.map(get_embeddings, batched=True)

Map:   0%|          | 0/18184 [00:00<?, ? examples/s]

32/32 ━━━━━━━━━━━━━━━━━━━━ 17s 280ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step


In [13]:
ds.shape

(18184, 3)

In [14]:
embeddings = np.array([e for e in ds['embedding']])
labels = np.array([l for l in ds['label']])

indices = np.arange(len(embeddings))
np.random.shuffle(indices)

embeddings_shuffled = embeddings[indices]
labels_shuffled = labels[indices]

pd.DataFrame(embeddings_shuffled).to_csv('embeddings.csv', index=False, header=False)
pd.DataFrame(labels_shuffled).to_csv('labels.csv', index=False, header=False)
